# Двухстадийный поиск по описаниям уязвимостей CVE с дообучением реранкера

Так как изначально код для проекта был написан в виде python-скриптов, этот jupyter-notebook

представляет собой объединение этих файлов (и не пересчитывает данные, есди они уже лежат в файлах)

## Общая конфигурация

Импорты, выбор устройства, пути и флаг `FORCE_RERUN`, общий для всех стадий.

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch


FORCE_RERUN = False

def cached(path: str) -> bool:
    return (not FORCE_RERUN) and Path(path).exists()

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"torch {torch.__version__}, device: {DEVICE}")

CORPUS_FILE = "dataset/corpus.json"
QUERIES_FILE = "dataset/queries.json"
TRAIN_FILE = "dataset/prepared_data/queries_train.json"
TEST_FILE = "dataset/prepared_data/queries_test.json"

os.makedirs("runs", exist_ok=True)
os.makedirs("models", exist_ok=True)

torch 2.12.0, device: mps


## 1. Подготовка данных

Корпус — описания уязвимостей из датасета [CVEfixes](https://huggingface.co/datasets/hitoshura25/cvefixes).
Из каждой записи берём `cve_id`, описание (`text`) и `cwe_name`; отбрасываем дубликаты
и слишком короткие описания, перемешиваем с фиксированным сидом и оставляем 4500 документов.

In [ ]:
from datasets import load_dataset
import json, random
import ast

if cached(CORPUS_FILE):
    corpus = [json.loads(l) for l in open(CORPUS_FILE, encoding="utf-8")]
    print(f"{CORPUS_FILE} уже готов: {len(corpus)} документов — пропускаем подготовку")
else:
    ds = load_dataset("hitoshura25/cvefixes", split="train")

    # print(ast.literal_eval(ds[0].get("cve_description"))[0].get('value'))

    seen = set()
    corpus = []
    for row in ds:
        cve_id = row.get("cve_id")
        desc = ast.literal_eval(row.get("cve_description"))[0].get('value')

        if cve_id in seen or not desc or len(desc) < 50:
            continue

        seen.add(cve_id)
        corpus.append({
            "cve_id": cve_id,
            "text": desc.strip(),
            "cwe_name": row.get("cwe_name")
        })

    random.seed(42)
    random.shuffle(corpus)
    corpus = corpus[:4500]

    with open(CORPUS_FILE, "w") as f:
        for d in corpus:
            f.write(json.dumps(d) + "\n")

    print(f"Saved {len(corpus)} documents")

dataset/corpus.json уже готов: 4500 документов — пропускаем подготовку


## 1.1 Генерация синтетических запросов (doc2query)

Размеченных пар запрос - релевантный документ для области ИБ нет, поэтому запросы
генерируются автоматически моделью `BeIR/query-gen-msmarco-t5-base-v1`: на каждый
документ генерируется 4 кандидата, из которых после фильтрации (длина,
вхождение в документ, дедупликация) оставляется 2. Генерация идёт с
`do_sample=True, top_p=0.95` для разнообразия. Скрипт умеет дозаписывать: уже
обработанные `cve_id` пропускаются.

In [ ]:
import json
from pathlib import Path

import torch
from tqdm import tqdm
from transformers import T5ForConditionalGeneration, T5Tokenizer

INPUT_FILE = "dataset/corpus.json"
OUTPUT_FILE = "dataset/queries.json"
MODEL_NAME = "BeIR/query-gen-msmarco-t5-base-v1"
BATCH_SIZE = 8
N_CANDIDATES = 4 # for gen
N_KEEP = 2 # after filtering
MAX_INPUT_LEN = 384
MAX_QUERY_LEN = 64

device = DEVICE
print(f"Device: {device}")


def is_bad_query(q: str, doc_text: str) -> bool:
    words = q.split()
    if not (3 <= len(words) <= 25):
        return True
    if q.lower() in doc_text.lower():
        return True

    return False


def pick_diverse(candidates, doc_text, n_keep):
    seen_norm, picked = set(), []
    for q in candidates:
        q = q.strip()
        norm = " ".join(sorted(set(q.lower().split())))
        if is_bad_query(q, doc_text) or norm in seen_norm:
            continue

        seen_norm.add(norm)
        picked.append(q)
        if len(picked) == n_keep:
            break

    return picked


@torch.no_grad()
def generate_batch(texts):
    inputs = tokenizer(
        texts, return_tensors="pt", truncation=True,
        max_length=MAX_INPUT_LEN, padding=True,
    ).to(device)

    outs = model.generate(
        **inputs,
        max_length=MAX_QUERY_LEN,
        do_sample=True,
        top_p=0.95,
        temperature=1.0,
        num_return_sequences=N_CANDIDATES,
    )

    decoded = tokenizer.batch_decode(outs, skip_special_tokens=True)

    return [decoded[i * N_CANDIDATES:(i + 1) * N_CANDIDATES] for i in range(len(texts))]


def main():
    corpus = [json.loads(l) for l in open(INPUT_FILE, encoding="utf-8")]

    done_ids = set()
    if Path(OUTPUT_FILE).exists():
        for line in open(OUTPUT_FILE, encoding="utf-8"):
            try:
                done_ids.add(json.loads(line)["cve_id"])
            except json.JSONDecodeError:
                pass

        print(f"already processed: {len(done_ids)}")

    todo = [d for d in corpus if d["cve_id"] not in done_ids]
    print(f"{len(todo)} docs needs to be processed")

    skipped = 0
    with open(OUTPUT_FILE, "a", buffering=1, encoding="utf-8") as out:
        for i in tqdm(range(0, len(todo), BATCH_SIZE), desc="doc2query"):
            batch = todo[i:i + BATCH_SIZE]
            candidates_per_doc = generate_batch([d["text"] for d in batch])

            for doc, candidates in zip(batch, candidates_per_doc):
                picked = pick_diverse(candidates, doc["text"], N_KEEP)
                if len(picked) < N_KEEP:
                    skipped += 1
                    continue

                record = {
                    "cve_id": doc["cve_id"],
                    "queries": [
                        {"query_id": f"{doc['cve_id']}__{j+1}",
                         "text": q, "style": "doc2query"}
                        for j, q in enumerate(picked)
                    ],
                }
                out.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Done. Skipped {skipped}")


if cached(OUTPUT_FILE):
    n = sum(1 for _ in open(OUTPUT_FILE, encoding="utf-8"))
    print(f"{OUTPUT_FILE} is ready: {n} записей — skip doc2query")
else:
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
    model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device).eval()
    main()

Device: mps
dataset/queries.json is ready: 4331 записей — skip doc2query


## 1.2 Разделение на train / test

Сплит по идентификаторам документов (а не по запросам): оба запроса одного CVE
попадают в одну часть — это исключает утечку. 20 % документов идут в тест, 80 % в трейн.

In [ ]:
import json
import random
from pathlib import Path

CORPUS_FILE = "dataset/corpus.json"
QUERIES_FILE = "dataset/queries.json"
OUT_DIR = Path("dataset/prepared_data")
SEED = 1
TEST_SHARE_PERC = 0.2 # 20% to 80% for train


def main():
    OUT_DIR.mkdir(exist_ok=True)

    corpus = [json.loads(l) for l in open(CORPUS_FILE, encoding="utf-8")]
    raw_queries = [json.loads(l) for l in open(QUERIES_FILE, encoding="utf-8")]
    known_ids = {d["cve_id"] for d in corpus}

    queries = []
    for item in raw_queries:
        if item["cve_id"] not in known_ids:
            continue
        for q in item["queries"]:
            queries.append({
                "query_id": q["query_id"],
                "text": q["text"],
                "target_doc": item["cve_id"],
            })
    print(f"doc len: {len(corpus)}, q len: {len(queries)}")

    doc_ids = sorted({q["target_doc"] for q in queries})

    random.Random(SEED).shuffle(doc_ids)

    test_docs = set(doc_ids[:int(len(doc_ids) * TEST_SHARE_PERC)])

    train = [q for q in queries if q["target_doc"] not in test_docs]
    test = [q for q in queries if q["target_doc"] in test_docs]

    for name, qs in [("train", train), ("test", test)]:
        with open(OUT_DIR / f"queries_{name}.json", "w", encoding="utf-8") as f:
            for q in qs:
                f.write(json.dumps(q, ensure_ascii=False) + "\n")

        print(f"{name}: {len(qs)} queries, {len({q['target_doc'] for q in qs})} docs")


if cached(TRAIN_FILE) and cached(TEST_FILE):
    print("train/test splitis ready — skip splitting")
else:
    main()

train/test splitis ready — skip splitting


## 2–3. Первая стадия (BM25, bi-encoder) и zero-shot реранк

Две архитектуры первой стадии: лексический **BM25** и плотный **bi-encoder**
(`intfloat/e5-base-v2`, косинусная близость нормированных эмбеддингов). Каждая
отбирает по 100 кандидатов на запрос. Затем недообученный (zero-shot) cross-encoder
`cross-encoder/ms-marco-MiniLM-L-12-v2` переранжирует этих кандидатов.

Получаем 4 конфигурации и сохраняем их в `runs/`: `bm25`, `bi-encoder`,
`bm25 + ce`, `bi-encoder + ce`.

In [ ]:
import pyarrow.dataset # без этого segfault на windows
import json
import time
import os

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from ranx import Qrels, Run, evaluate


os.makedirs("runs", exist_ok=True)

CORPUS_FILE = "dataset/corpus.json"
TEST_FILE = "dataset/prepared_data/queries_test.json"
TOP_K = 100  # candidate for second stage

corpus = [json.loads(l) for l in open(CORPUS_FILE, encoding="utf-8")]
doc_ids = [d["cve_id"] for d in corpus]
doc_texts = [d["text"] for d in corpus]
text_by_id = dict(zip(doc_ids, doc_texts))

test_q = [json.loads(l) for l in open(TEST_FILE, encoding="utf-8")]

# for metrics
qrels = Qrels({q["query_id"]: {q["target_doc"]: 1} for q in test_q})

print(f"Docs: {len(doc_ids)}, Test queries: {len(test_q)}")

BASELINE_RUNS = {
    "bm25": "runs/bm25.json",
    "bi-encoder": "runs/bi-encoder.json",
    "bm25 + ce": "runs/bm25_+_ce.json",
    "bi-encoder + ce": "runs/bi-encoder_+_ce.json",
}

if all(cached(p) for p in BASELINE_RUNS.values()):
    print("\nbaseline runs is ready - load from disk")
    bm25_run = Run.from_file("runs/bm25.json").to_dict()
    dense_run = Run.from_file("runs/bi-encoder.json").to_dict()
    bm25_ce_run = Run.from_file("runs/bm25_+_ce.json").to_dict()
    dense_ce_run = Run.from_file("runs/bi-encoder_+_ce.json").to_dict()
else:
    # BM25
    print("\nBM25: make index...")
    bm25 = BM25Okapi([t.lower().split() for t in doc_texts])

    bm25_run = {}
    t0 = time.time()
    for q in test_q:
        scores = bm25.get_scores(q["text"].lower().split())
        top = np.argsort(scores)[::-1][:TOP_K]
        bm25_run[q["query_id"]] = {doc_ids[i]: float(scores[i]) for i in top}
    print(f"search: {1000 * (time.time() - t0) / len(test_q):.1f} ms for q")

    # bi-encoder
    encoder = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE)

    print("\nbi-encoder: offline")
    doc_emb = encoder.encode(["passage: " + t for t in doc_texts], batch_size=64, normalize_embeddings=True, show_progress_bar=True)

    print("\nbi-encoder: online")
    dense_run = {}
    t0 = time.time()
    for q in test_q:
        q_embed = encoder.encode("query: " + q["text"], normalize_embeddings=True) # обычно запрос приходит уже в онлайне, поэтому честно время так замерить
        scores = doc_emb @ q_embed  # векторы нормированы поэтому cкалярное произведение = кос. близость
        top = np.argsort(scores)[::-1][:TOP_K]
        dense_run[q["query_id"]] = {doc_ids[i]: float(scores[i]) for i in top}
    print(f"search: {1000 * (time.time() - t0) / len(test_q):.2f} ms for q")

    # Rerank
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", max_length=320, device=DEVICE)

    def rerank(first_stage_run):
        # вторая стадия видит только top k кандидатов
        new_run = {}
        times = []
        for q in test_q:
            candidates = list(first_stage_run[q["query_id"]].keys())
            pairs = [(q["text"], text_by_id[d]) for d in candidates]
            t0 = time.time()
            scores = reranker.predict(pairs, batch_size=64)
            times.append(time.time() - t0)
            new_run[q["query_id"]] = dict(zip(candidates, map(float, scores)))
        print(f"rerank: {1000 * np.mean(times):.0f} ms for q")
        return new_run

    print("\nrerank BM25...")
    bm25_ce_run = rerank(bm25_run)
    print("rerank bi-encoder...")
    dense_ce_run = rerank(dense_run)

    for name, run_dict in {
        "bm25": bm25_run,
        "bi-encoder": dense_run,
        "bm25 + ce": bm25_ce_run,
        "bi-encoder + ce": dense_ce_run,
    }.items():
        Run(run_dict, name=name).save(f"runs/{name.replace(' ', '_')}.json")
    print("saved baseline runs")

Docs: 4500, Test queries: 1732

baseline runs is ready - load from disk


Оценка четырёх базовых конфигураций (метрики считаются по сохранённым ранам):

In [ ]:
runs = {
    "bm25": bm25_run,
    "bi-encoder": dense_run,
    "bm25 + ce": bm25_ce_run,
    "bi-encoder + ce": dense_ce_run,
}

metrics = ["recall@100", "recall@10", "ndcg@10", "mrr@10"]

print("\n" + "-" * 70)
print(f"{'config':<18}" + "".join(f"{m:>13}" for m in metrics))
print("-" * 70)

for name, run_dict in runs.items():
    res = evaluate(qrels, Run(run_dict), metrics)
    print(f"{name:<18}" + "".join(f"{res[m]:>13.4f}" for m in metrics))


----------------------------------------------------------------------
config               recall@100    recall@10      ndcg@10       mrr@10
----------------------------------------------------------------------
bm25                     0.7038       0.5560       0.4417       0.4053
bi-encoder               0.9613       0.8418       0.7083       0.6657
bm25 + ce                0.7038       0.6490       0.5704       0.5450
bi-encoder + ce          0.9613       0.8672       0.7479       0.7095


## 4. Подготовка обучающих данных и дообучение реранкера (наивное)

Hard negatives добываются bi-encoder'ом: для каждого запроса берём наиболее
похожие train-документы, исключая правильный. Cross-encoder дообучается в режиме
бинарной классификации релевантности (`num_labels=1`, метки 0/1 → BCE).

Эта версия — **без** consistency filtering: 4 негатива на позитив, 1 эпоха.
Если модель `models/ce_finetuned` уже обучена — загружаем её, обучение пропускаем.

In [ ]:
import pyarrow.dataset
import json
import os
import random
import time

import numpy as np
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from ranx import Qrels, Run, evaluate

CORPUS_FILE = "dataset/corpus.json"
TRAIN_FILE = "dataset/prepared_data/queries_train.json"
TEST_FILE = "dataset/prepared_data/queries_test.json"
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"
OUT_MODEL = "models/ce_finetuned"
N_NEGATIVES = 4   # per one positive
EPOCHS = 1
BATCH = 16
LR = 2e-5
SEED = 1

random.seed(SEED)
os.makedirs("models", exist_ok=True)
os.makedirs("runs", exist_ok=True)

corpus = [json.loads(l) for l in open(CORPUS_FILE, encoding="utf-8")]
text_by_id = {d["cve_id"]: d["text"] for d in corpus}

train_q = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8")]
test_q = [json.loads(l) for l in open(TEST_FILE, encoding="utf-8")]

if cached(OUT_MODEL):
    print(f"{OUT_MODEL} уже обучен — загружаем, обучение пропускаем")
    model = CrossEncoder(OUT_MODEL, max_length=320, device=DEVICE)
else:
    # пул для негативов, только train-документы, чтобы тексты теста не участвовали в обучении
    train_doc_ids = sorted({q["target_doc"] for q in train_q})
    train_doc_texts = [text_by_id[d] for d in train_doc_ids]
    print(f"train-q: {len(train_q)}, doc len for negatives: {len(train_doc_ids)}")

    # негативы

    encoder = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE)
    print("encode train-docs...")
    doc_emb = encoder.encode(["passage: " + t for t in train_doc_texts],
                             batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    print("ecnode train-queries...")
    q_emb = encoder.encode(["query: " + q["text"] for q in train_q],
                           batch_size=64, normalize_embeddings=True, show_progress_bar=True)

    examples = []
    for q, emb in zip(train_q, q_emb):
        top = np.argsort(doc_emb @ emb)[::-1][:N_NEGATIVES + 10] # используем тот же bi-encoder
        negatives = [
            train_doc_ids[i] for i in top
                if train_doc_ids[i] != q["target_doc"]][:N_NEGATIVES] # и исключаем из негативов таргетный док
        examples.append(InputExample(texts=[q["text"], text_by_id[q["target_doc"]]], label=1.0))
        for neg in negatives:
            examples.append(InputExample(texts=[q["text"], text_by_id[neg]], label=0.0))

    random.shuffle(examples)
    print(f"train pairs: {len(examples)}, positive {len(train_q)}, neg {len(examples) - len(train_q)})")

    model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=320, device=DEVICE)
    loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
    n_steps = len(loader) * EPOCHS

    t0 = time.time()
    model.fit(
        train_dataloader=loader,
        epochs=EPOCHS,
        warmup_steps=int(0.1 * n_steps),
        optimizer_params={"lr": LR},
        show_progress_bar=True,
    )
    print(f"learning time: {(time.time() - t0) / 60:.1f} мин")
    model.save(OUT_MODEL)
    print(f"model saved {OUT_MODEL}")

models/ce_finetuned уже обучен — загружаем, обучение пропускаем


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Реранжируем дообученной моделью **те же** кандидаты первой стадии, что и в
бейзлайне (`runs/bm25.json`, `runs/bi-encoder.json`), и сохраняем 2 новых рана.

In [ ]:
qrels = Qrels({q["query_id"]: {q["target_doc"]: 1} for q in test_q})

def rerank(first_stage_run, ce):
    new_run = {}
    for q in test_q:
        candidates = list(first_stage_run[q["query_id"]].keys())
        pairs = [(q["text"], text_by_id[d]) for d in candidates]
        scores = ce.predict(pairs, batch_size=64)
        new_run[q["query_id"]] = dict(zip(candidates, map(float, scores)))
    return new_run

FT_RUNS = ["runs/bm25_ce_ft.json", "runs/bi-encoder_ce_ft.json"]
if all(cached(p) for p in FT_RUNS):
    print("ce-ft runs уже готовы — реранк пропускаем")
else:
    print("\nrerank with fine-tuned model")
    bm25_run = Run.from_file("runs/bm25.json").to_dict()
    dense_run = Run.from_file("runs/bi-encoder.json").to_dict()

    bm25_ft_run = rerank(bm25_run, model)
    Run(bm25_ft_run, name="bm25+ce-ft").save("runs/bm25_ce_ft.json")
    dense_ft_run = rerank(dense_run, model)
    Run(dense_ft_run, name="bi-encoder+ce-ft").save("runs/bi-encoder_ce_ft.json")

ce-ft runs уже готовы — реранк пропускаем


Сводная таблица: 4 базовые конфигурации + 2 с наивно дообученным реранкером.

In [ ]:
all_runs = {
    "bm25":               "runs/bm25.json",
    "bi-encoder":         "runs/bi-encoder.json",
    "bm25 + ce":          "runs/bm25_+_ce.json",
    "bi-encoder + ce":    "runs/bi-encoder_+_ce.json",
    "bm25 + ce-ft":       "runs/bm25_ce_ft.json",
    "bi-encoder + ce-ft": "runs/bi-encoder_ce_ft.json",
}
metrics = ["recall@100", "recall@10", "ndcg@10", "mrr@10"]

print("\n" + "-" * 75)
print(f"{'config':<22}" + "".join(f"{m:>13}" for m in metrics))
print("-" * 75)
for name, path in all_runs.items():
    res = evaluate(qrels, Run.from_file(path), metrics)
    print(f"{name:<22}" + "".join(f"{res[m]:>13.4f}" for m in metrics))


---------------------------------------------------------------------------
config                   recall@100    recall@10      ndcg@10       mrr@10
---------------------------------------------------------------------------
bm25                         0.7038       0.5589       0.4434       0.4065
bi-encoder                   0.9613       0.8418       0.7083       0.6657
bm25 + ce                    0.7038       0.6490       0.5704       0.5450
bi-encoder + ce              0.9613       0.8672       0.7479       0.7095
bm25 + ce-ft                 0.7038       0.6524       0.5745       0.5492
bi-encoder + ce-ft           0.9613       0.8643       0.7491       0.7118


## 4.1 Дообучение с consistency filtering

То же дообучение, но с **consistency filtering обучающих запросов**:
запрос остаётся, только если bi-encoder возвращает его исходный документ в топ-10.
Это убирает неспецифичные синтетические запросы, вносящие шум. Здесь усилены
гиперпараметры: 6 негативов на позитив, 2 эпохи. Модель — `models/ce_finetuned_const_filt`.

In [ ]:
import pyarrow.dataset
import json
import os
import random
import time

import numpy as np
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from ranx import Qrels, Run, evaluate

CORPUS_FILE = "dataset/corpus.json"
TRAIN_FILE = "dataset/prepared_data/queries_train.json"
TEST_FILE = "dataset/prepared_data/queries_test.json"
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"
OUT_MODEL = "models/ce_finetuned_const_filt"
N_NEGATIVES = 6 # up
EPOCHS = 2 # up
BATCH = 16
LR = 2e-5
SEED = 1

random.seed(SEED)
os.makedirs("models", exist_ok=True)
os.makedirs("runs", exist_ok=True)

corpus = [json.loads(l) for l in open(CORPUS_FILE, encoding="utf-8")]
text_by_id = {d["cve_id"]: d["text"] for d in corpus}

train_q = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8")]
test_q = [json.loads(l) for l in open(TEST_FILE, encoding="utf-8")]

if cached(OUT_MODEL):
    print(f"{OUT_MODEL} уже обучен — загружаем, обучение пропускаем")
    model = CrossEncoder(OUT_MODEL, max_length=320, device=DEVICE)
else:
    train_doc_ids = sorted({q["target_doc"] for q in train_q})
    train_doc_texts = [text_by_id[d] for d in train_doc_ids]
    print(f"train-q: {len(train_q)}, doc len for negatives: {len(train_doc_ids)}")

    encoder = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE)
    print("encode train-docs...")
    doc_emb = encoder.encode(["passage: " + t for t in train_doc_texts],
                             batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    print("ecnode train-queries...")
    q_emb = encoder.encode(["query: " + q["text"] for q in train_q],
                           batch_size=64, normalize_embeddings=True, show_progress_bar=True)

    examples = []

    print("consistency filtering train-q...")
    kept = []
    for q, emb in zip(train_q, q_emb):
        top10 = np.argsort(doc_emb @ emb)[::-1][:10] # оставляем запрос, только если bi-encoder вернул в топ 10
        if q["target_doc"] in [train_doc_ids[i] for i in top10]:
            kept.append(q)
    print(f"kept {len(kept)}/{len(train_q)} after filtration")
    train_q = kept

    for q, emb in zip(train_q, q_emb):
        top = np.argsort(doc_emb @ emb)[::-1][:N_NEGATIVES + 10]
        negatives = [train_doc_ids[i] for i in top
                     if train_doc_ids[i] != q["target_doc"]][:N_NEGATIVES]
        examples.append(InputExample(texts=[q["text"], text_by_id[q["target_doc"]]], label=1.0))
        for neg in negatives:
            examples.append(InputExample(texts=[q["text"], text_by_id[neg]], label=0.0))

    random.shuffle(examples)
    print(f"train pairs: {len(examples)}, positive {len(train_q)}, neg {len(examples) - len(train_q)})")

    model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=320, device=DEVICE)
    loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
    n_steps = len(loader) * EPOCHS

    t0 = time.time()
    model.fit(
        train_dataloader=loader,
        epochs=EPOCHS,
        warmup_steps=int(0.1 * n_steps),
        optimizer_params={"lr": LR},
        show_progress_bar=True,
    )
    print(f"learning time: {(time.time() - t0) / 60:.1f} мин")
    model.save(OUT_MODEL)
    print(f"model saved {OUT_MODEL}")

models/ce_finetuned_const_filt уже обучен — загружаем, обучение пропускаем


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Реранжируем те же кандидаты первой стадии моделью с consistency filtering
и сохраняем 2 рана.

In [ ]:
qrels = Qrels({q["query_id"]: {q["target_doc"]: 1} for q in test_q})

def rerank(first_stage_run, ce):
    new_run = {}
    for q in test_q:
        candidates = list(first_stage_run[q["query_id"]].keys())
        pairs = [(q["text"], text_by_id[d]) for d in candidates]
        scores = ce.predict(pairs, batch_size=64)
        new_run[q["query_id"]] = dict(zip(candidates, map(float, scores)))
    return new_run

CF_RUNS = ["runs/bm25_ce_ft_const_filt.json", "runs/bi-encoder_ce_ft_const_filt.json"]
if all(cached(p) for p in CF_RUNS):
    print("ce-ft-cf runs уже готовы — реранк пропускаем")
else:
    print("\nrerank with fine-tuned model")
    bm25_run = Run.from_file("runs/bm25.json").to_dict()
    dense_run = Run.from_file("runs/bi-encoder.json").to_dict()

    bm25_ft_run = rerank(bm25_run, model)
    Run(bm25_ft_run, name="bm25+ce-ft-cf").save("runs/bm25_ce_ft_const_filt.json")
    dense_ft_run = rerank(dense_run, model)
    Run(dense_ft_run, name="bi-encoder+ce-ft-cf").save("runs/bi-encoder_ce_ft_const_filt.json")

ce-ft-cf runs уже готовы — реранк пропускаем


Итоговая таблица всех 8 конфигураций.

In [ ]:
all_runs = {
    "bm25":                  "runs/bm25.json",
    "bi-encoder":            "runs/bi-encoder.json",
    "bm25 + ce":             "runs/bm25_+_ce.json",
    "bi-encoder + ce":       "runs/bi-encoder_+_ce.json",
    "bm25 + ce-ft":          "runs/bm25_ce_ft.json",
    "bi-encoder + ce-ft":    "runs/bi-encoder_ce_ft.json",
    "bm25 + ce-ft-cf":       "runs/bm25_ce_ft_const_filt.json",
    "bi-encoder + ce-ft-cf": "runs/bi-encoder_ce_ft_const_filt.json",
}
metrics = ["recall@100", "recall@10", "ndcg@10", "mrr@10"]

print("\n" + "-" * 75)
print(f"{'config':<22}" + "".join(f"{m:>13}" for m in metrics))
print("-" * 75)
for name, path in all_runs.items():
    res = evaluate(qrels, Run.from_file(path), metrics)
    print(f"{name:<22}" + "".join(f"{res[m]:>13.4f}" for m in metrics))


---------------------------------------------------------------------------
config                   recall@100    recall@10      ndcg@10       mrr@10
---------------------------------------------------------------------------
bm25                         0.7038       0.5589       0.4434       0.4065
bi-encoder                   0.9613       0.8418       0.7083       0.6657
bm25 + ce                    0.7038       0.6490       0.5704       0.5450
bi-encoder + ce              0.9613       0.8672       0.7479       0.7095
bm25 + ce-ft                 0.7038       0.6524       0.5745       0.5492
bi-encoder + ce-ft           0.9613       0.8643       0.7491       0.7118
bm25 + ce-ft-cf              0.7038       0.6547       0.5789       0.5544
bi-encoder + ce-ft-cf        0.9613       0.8753       0.7562       0.7177


## 5. Статистическая значимость

Парный тест Стьюдента по метрикам `ndcg@10` и `mrr@10` для четырёх ключевых конфигураций на основе bi-encoder.

In [ ]:
from ranx import Qrels, Run, compare
import json

test_q = [json.loads(l) for l in open("dataset/prepared_data/queries_test.json")]
qrels = Qrels({q["query_id"]: {q["target_doc"]: 1} for q in test_q})

names = ["bi-encoder", "bi-encoder_+_ce", "bi-encoder_ce_ft", "bi-encoder_ce_ft_const_filt"]
runs = [Run.from_file(f"runs/{n}.json") for n in names]

report = compare(qrels, runs, metrics=["ndcg@10", "mrr@10"],
                 max_p=0.05, stat_test="student")
print(report)

#    Model    NDCG@10    MRR@10
---  -------  ---------  --------
a    run_1    0.708      0.666
b    run_2    0.748ᵃ     0.710ᵃ
c    run_3    0.749ᵃ     0.712ᵃ
d    run_4    0.756ᵃᵇ    0.718ᵃ


## Выводы

- **Плотный поиск значительно превосходит лексический на первой стадии:**
  bi-encoder даёт Recall@100 ≈ 0.96 против ≈ 0.70 у BM25. Полнота первой стадии
  задаёт потолок всего конвейера, поэтому основной выбор — bi-encoder.
- **Двухстадийная схема значимо улучшает ранжирование:** добавление
  недообученного реранкера поднимает nDCG@10 с ≈ 0.708 до ≈ 0.748.
- **Дообучение реранкера эффективно только с consistency filtering:** версия с
  фильтрацией даёт nDCG@10 ≈ 0.756 (статистически значимо относительно zero-shot
  реранкера, p < 0.05), тогда как наивное дообучение (≈ 0.749) значимого прироста
  не даёт. По MRR@10 прирост значимости не достигает.

Ключевой фактор успеха доменной адаптации на синтетических данных — фильтрация
обучающих примеров по согласованности.